# Figure 5: Ultra-low-dimensional embeddings

Run after the training and evaluation commands in `bash/paper/`. SCENE outputs use the `scLDM` names referenced below.


In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd

PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "environment_scene.yaml").exists())
os.chdir(PROJECT_ROOT / "notebooks" / "figures")
for folder in ("fig_1", "fig_2", "fig_3", "fig_4", "fig_5", "fig_6", "app", "qc"):
    Path("output", folder).mkdir(parents=True, exist_ok=True)


In [ ]:
import anndata as ad
import numpy as np

adata = ad.read_h5ad("../../data/neurips_cite_gex_stem_cells.h5ad")

# Match the exported IDs: training can remove unexpressed genes.
adata = adata[
    np.load("../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_cell_names.npy"),
    np.load("../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_gene_names.npy"),
].copy()

z_cells_2d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_cell_latent.npy')
z_genes_2d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_2D/scLDM_gene_latent.npy')

if z_cells_2d.shape[1] != z_genes_2d.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_2d.shape}, genes={z_genes_2d.shape}")

adata.obsm['scLDM_2d'] = z_cells_2d
adata.varm['scLDM_2d'] = z_genes_2d


z_cells_3d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_3D/scLDM_cell_latent.npy')
z_genes_3d = np.load('../../results/neurips_cite_stem_cells/scLDM_batch_full_3D/scLDM_gene_latent.npy')

if z_cells_3d.shape[1] != z_genes_3d.shape[1]:
    raise ValueError(f"Latent dimension mismatch: cells={z_cells_3d.shape}, genes={z_genes_3d.shape}")

adata.obsm['scLDM_3d'] = z_cells_3d
adata.varm['scLDM_3d'] = z_genes_3d

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal


def _build_palette_for_series(series):
    categories = list(pd.Categorical(series.astype(str)).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


COMMON_CATEGORIES = {}
COMMON_PALETTES = {}
for _label in ["cell_type", "label"]:
    if _label in adata.obs.columns:
        _cats, _pal = _build_palette_for_series(adata.obs[_label])
        COMMON_CATEGORIES[_label] = _cats
        COMMON_PALETTES[_label] = _pal



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import numpy as np

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


Z_cells = np.asarray(adata.obsm["scLDM_3d"], dtype=float)
Z_genes = np.asarray(adata.varm["scLDM_3d"], dtype=float)

if Z_cells.shape[1] != 3 or Z_genes.shape[1] != 3:
    raise ValueError(f"Expected 3D embeddings. Got cells={Z_cells.shape}, genes={Z_genes.shape}")

def build_palette(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)
    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]
    return categories, dict(zip(categories, colors))

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

views = [("angle1", 20, 10), ("angle2", 20, 30), ("angle3", 30, 40), ("angle4", 10, 315)]

outdir = Path("output/fig_5")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette(values)

    for view_name, elev, azim in views:
        fig = plt.figure(figsize=(2, 2))
        ax = fig.add_subplot(111, projection="3d")

        for ct in categories:
            idx = values_np == ct
            if np.any(idx):
                ax.scatter(
                    Z_cells[idx, 0], Z_cells[idx, 1], Z_cells[idx, 2],
                    s=0.3, c=[palette[ct]], alpha=1, linewidths=0, depthshade=False, rasterized=True
                )

        ax.scatter(
            Z_genes[:, 0], Z_genes[:, 1], Z_genes[:, 2],
            s=0.1, c="#7A1E1E", alpha=.5, linewidths=0, depthshade=False, rasterized=True 
        )

        ax.view_init(elev=elev, azim=azim)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.tick_params(axis="x", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="y", which="major", labelsize=4, length=2, pad=0)
        ax.tick_params(axis="z", which="major", labelsize=4, length=2, pad=0)

        ax.set_xlim(-5, 5)
        ax.set_ylim(-5, 5)
        ax.set_zlim(-3, 3)
        
        ax.set_position([0.02, 0.02, 0.96, 0.97])
        ax.set_box_aspect((
            np.ptp(ax.get_xlim()),
            np.ptp(ax.get_ylim()),
            np.ptp(ax.get_zlim())
        ))
        fig.subplots_adjust(left=0.0, right=1.00, bottom=0.00, top=1.0)

        plt.savefig(
            outdir / f"kang_scLDM_3d_{label}_{view_name}.svg",
            bbox_inches="tight",
            pad_inches=0,
            dpi=600,
        )
        plt.show()


## 2D Native

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from matplotlib.lines import Line2D
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import numpy as np


def build_palette_local(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


Z_cells_2d = np.asarray(adata.obsm["scLDM_2d"], dtype=float)
Z_genes_2d = np.asarray(adata.varm["scLDM_2d"], dtype=float)
if Z_cells_2d.shape[1] != 2 or Z_genes_2d.shape[1] != 2:
    raise ValueError(
        f"Expected 2D embeddings in scLDM_2d. Got cells={Z_cells_2d.shape}, genes={Z_genes_2d.shape}"
    )

Z_cells_3d = np.asarray(adata.obsm["scLDM_3d"], dtype=float)
Z_genes_3d = np.asarray(adata.varm["scLDM_3d"], dtype=float)
if Z_cells_3d.shape[1] != 3 or Z_genes_3d.shape[1] != 3:
    raise ValueError(
        f"Expected 3D embeddings in scLDM_3d. Got cells={Z_cells_3d.shape}, genes={Z_genes_3d.shape}"
    )

labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [col for col in ["cell_type", "label"] if col in adata.obs.columns]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata.obs")

outdir = Path("output/fig_5")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    values_np = adata.obs[label].astype(str).to_numpy()
    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        values = adata.obs[label].astype(str)
        categories, palette = build_palette_local(values)

    # -------- 2D static (fig_3 style) --------
    fig, ax = plt.subplots(figsize=(2., 2.))

    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                Z_cells_2d[idx, 0],
                Z_cells_2d[idx, 1],
                s=0.3,
                c=[palette[ct]],
                alpha=1,
                linewidths=0,
                rasterized=True
            )

    ax.scatter(
        Z_genes_2d[:, 0],
        Z_genes_2d[:, 1],
        s=0.1,
        c="#7A1E1E",
        alpha=0.5,
        linewidths=0,
        rasterized=True
    )

    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(
        outdir / f"kang_scLDM_2d_{label}.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600,
    )
    plt.show()
    # plt.close(fig)


## 2D UMAP

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scanpy as sc

# ----- 1.  Build a placeholder expression matrix for the genes -----
# Gene rows have zero-filled expression values; their embeddings are stored separately.
n_cells, n_genes = adata.n_obs, adata.n_vars
if sp.issparse(adata.X):
    placeholder_X = sp.csr_matrix((n_genes, n_genes), dtype=adata.X.dtype)
else:
    placeholder_X = np.zeros((n_genes, n_genes), dtype=adata.X.dtype)

# ----- 2.  Stack the real cells with the pseudo-cells (genes) -----
X_combined = (
    sp.vstack([adata.X, placeholder_X], format="csr")
    if sp.issparse(adata.X)
    else np.vstack([adata.X, placeholder_X])
)

# ----- 3.  Create an .obs that labels each row as cell / gene -----
obs_combined = pd.concat(
    [
        adata.obs.assign(entity="cell"),               # keep existing cell metadata
        pd.DataFrame({"entity": "gene"}, index=adata.var_names)  # one row per gene
    ]
)

# ----- 4.  Assemble the new AnnData object -----
adata_combo = sc.AnnData(
    X=X_combined,
    obs=obs_combined,
    var=adata.var.copy()            # keep original gene metadata as .var
)

# ----- 5.  Concatenate the embeddings and store in .obsm -----

adata_combo.obsm["scLDM_3d"] = np.vstack([
    adata.obsm["scLDM_3d"],      # cells (n_cells × dim)
    adata.varm["scLDM_3d"]       # genes (n_genes × dim)
])


In [ ]:
import scanpy as sc

sc.pp.neighbors(adata_combo, use_rep="scLDM_3d")
sc.tl.umap(adata_combo, key_added="umap_scLDM_3d")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from scanpy.plotting import palettes as scpal
import pandas as pd
from pathlib import Path
import numpy as np

# 2D UMAP on adata_combo (cells + gene pseudo-points), like fig_3.
if "adata_combo" not in globals():
    raise ValueError("adata_combo not found. Run the cell that builds adata_combo first.")

if "umap_scLDM_3d" in adata_combo.obsm:
    umap = np.asarray(adata_combo.obsm["umap_scLDM_3d"], dtype=float)
elif "X_umap" in adata_combo.obsm:
    umap = np.asarray(adata_combo.obsm["X_umap"], dtype=float)
else:
    raise ValueError(
        "UMAP not found in adata_combo. Run sc.pp.neighbors(adata_combo, use_rep='scLDM_3d') and sc.tl.umap(...)."
    )

if umap.shape[1] != 2:
    raise ValueError(f"Expected 2D UMAP coordinates, got shape={umap.shape}")

if "entity" not in adata_combo.obs.columns:
    raise ValueError("adata_combo.obs['entity'] is missing.")

entity = adata_combo.obs["entity"].astype(str)
cell_mask = entity.eq("cell").to_numpy()
gene_mask = entity.eq("gene").to_numpy()

cells = umap[cell_mask]
genes = umap[gene_mask]


def build_palette_umap(series):
    categories = list(pd.Categorical(series).categories)
    n_cat = len(categories)

    if n_cat <= 20:
        colors = scpal.default_20[:n_cat]
    elif n_cat <= 28:
        colors = scpal.default_28[:n_cat]
    elif n_cat <= 102:
        colors = scpal.default_102[:n_cat]
    else:
        cmap = plt.get_cmap("gist_ncar", n_cat)
        colors = [to_hex(cmap(i)) for i in range(n_cat)]

    return categories, dict(zip(categories, colors))


labels = list(COMMON_PALETTES.keys()) if "COMMON_PALETTES" in globals() else [
    col for col in ["cell_type", "label"] if col in adata_combo.obs.columns
]
if not labels:
    raise ValueError("Expected at least one of ['cell_type', 'label'] in adata_combo.obs")

outdir = Path("output/fig_5")
outdir.mkdir(parents=True, exist_ok=True)

for label in labels:
    if label not in adata_combo.obs.columns:
        continue

    values_np = adata_combo.obs.loc[cell_mask, label].astype(str).to_numpy()

    if "COMMON_CATEGORIES" in globals() and label in COMMON_CATEGORIES:
        categories = COMMON_CATEGORIES[label]
        palette = COMMON_PALETTES[label]
    else:
        categories, palette = build_palette_umap(adata_combo.obs.loc[cell_mask, label].astype(str))

    fig, ax = plt.subplots(figsize=(2, 2))

    for ct in categories:
        idx = values_np == ct
        if np.any(idx):
            ax.scatter(
                cells[idx, 0],
                cells[idx, 1],
                s=0.3,
                c=[palette[ct]],
                alpha=1,
                linewidths=0,
                rasterized=True
            )

    # overlay gene pseudo-points
    if genes.shape[0] > 0:
        ax.scatter(
            genes[:, 0],
            genes[:, 1],
            s=0.1,
            c="#7A1E1E",
            alpha=0.5,
            linewidths=0,
            rasterized=True
        )

    ax.set_axis_off()
    ax.set_title("")

    plt.savefig(
        outdir / f"kang_umap_2d_combo_{label}.svg",
        bbox_inches="tight",
        pad_inches=0,
        dpi=600,
    )
    plt.show()


In [ ]:
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from pathlib import Path
import pandas as pd


def save_cluster_legend_pdf(
    categories,
    palette,
    output_pdf,
    ncol=1,
    fontsize=6,
    marker_size=4.0,
    columnspacing=0.8,
    handletextpad=0.4,
    labelspacing=0.35,
    borderpad=0.2,
):
    """
    Save a standalone legend-only PDF for cluster colors.
    """

    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
    })


    handles = [
        Line2D(
            [0], [0],
            linestyle="None",
            marker="o",
            markersize=marker_size,
            markerfacecolor=palette[cat],
            markeredgecolor=palette[cat],
            markeredgewidth=0.0,
            label=str(cat),
        )
        for cat in categories
    ]

    n_items = len(categories)
    n_rows = math.ceil(n_items / ncol)
    fig_w = max(1.2, 1.15 * ncol + 0.55 * ncol)
    fig_h = max(0.35, 0.22 * n_rows + 0.18)

    fig = plt.figure(figsize=(fig_w, fig_h))
    fig.legend(
        handles=handles,
        labels=categories,
        loc="center",
        ncol=ncol,
        frameon=False,
        fontsize=fontsize,
        handlelength=0.8,
        handletextpad=handletextpad,
        columnspacing=columnspacing,
        labelspacing=labelspacing,
        borderpad=borderpad,
        markerscale=1.0,
    )

    fig.savefig(
        output_pdf,
        format="svg",
        bbox_inches="tight",
        pad_inches=0.01,
        transparent=True,
    )
    plt.close(fig)


outdir = Path("output/fig_5")
outdir.mkdir(parents=True, exist_ok=True)

# Prefer shared palettes built earlier; fallback to deriving from adata/adata_combo.
if "COMMON_CATEGORIES" in globals() and "COMMON_PALETTES" in globals() and COMMON_PALETTES:
    categories_by_label = {k: list(v) for k, v in COMMON_CATEGORIES.items()}
    palettes_by_label = {k: dict(v) for k, v in COMMON_PALETTES.items()}
else:
    source = adata_combo if "adata_combo" in globals() else adata
    categories_by_label = {}
    palettes_by_label = {}
    for label in ["cell_type", "label"]:
        if label in source.obs.columns:
            cats = list(pd.Categorical(source.obs[label].astype(str)).categories)
            if label == "cell_type" and "palette_celltypes" in globals():
                pal = palette_celltypes
            elif label == "label" and "palette_stim_groups" in globals():
                pal = palette_stim_groups
            else:
                # last-resort fallback if no palette variable exists
                pal = {c: "#808080" for c in cats}
            categories_by_label[label] = cats
            palettes_by_label[label] = pal


def append_genes_to_legend(categories, palette):
    categories_out = list(categories)
    palette_out = dict(palette)
    if "Genes" not in categories_out:
        categories_out.append("Genes")
    palette_out["Genes"] = "#7A1E1E"
    return categories_out, palette_out


if "cell_type" in categories_by_label:
    cats, pal = append_genes_to_legend(
        categories_by_label["cell_type"],
        palettes_by_label["cell_type"],
    )
    save_cluster_legend_pdf(
        categories=cats,
        palette=pal,
        output_pdf=str(outdir / "celltype_legend.svg"),
        ncol=1,
        fontsize=6,
        marker_size=4.0,
    )

if "label" in categories_by_label:
    cats, pal = append_genes_to_legend(
        categories_by_label["label"],
        palettes_by_label["label"],
    )
    save_cluster_legend_pdf(
        categories=cats,
        palette=pal,
        output_pdf=str(outdir / "label_legend.svg"),
        ncol=1,
        fontsize=6,
        marker_size=4.0,
    )


## 3D double gene scoring

In [ ]:
import numpy as np
import matplotlib as mpl

class SignedPowerNorm(mpl.colors.Normalize):
    def __init__(self, gamma=0.5, vmin=None, vmax=None, clip=False):
        super().__init__(vmin=vmin, vmax=vmax, clip=clip)
        self.gamma = gamma

    def __call__(self, value, clip=None):
        v = np.ma.asarray(value)
        self.autoscale_None(v)
        m = max(abs(self.vmin), abs(self.vmax))
        x = np.clip(v / m, -1, 1)                  # [-1, 1]
        y = np.sign(x) * (np.abs(x) ** self.gamma) # gamma on magnitude
        return 0.5 * (y + 1.0)                     # -> [0, 1]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})


# -------- pick two genes to contrast --------
gene_a = "FCGR3A"  # examples: LST1, FCGR3A, S100A9
gene_b = "HBB"    # examples: ALAS2, HBB, GYPA

if "adata_combo" not in globals():
    raise ValueError("adata_combo not found. Run the adata_combo construction cell first.")
if "scLDM_3d" not in adata_combo.obsm:
    raise ValueError("adata_combo.obsm['scLDM_3d'] not found.")
if "entity" not in adata_combo.obs.columns:
    raise ValueError("adata_combo.obs['entity'] is missing.")

Z = np.asarray(adata_combo.obsm["scLDM_3d"], dtype=float)
entity = adata_combo.obs["entity"].astype(str)
cell_mask = entity.eq("cell").to_numpy()
gene_mask = entity.eq("gene").to_numpy()

Z_cells = Z[cell_mask]
Z_genes = Z[gene_mask]
gene_names = adata_combo.obs_names[gene_mask].to_numpy().astype(str)

# find selected genes among gene pseudo-points
name_to_idx = {g: i for i, g in enumerate(gene_names)}
missing = [g for g in [gene_a, gene_b] if g not in name_to_idx]
if missing:
    raise ValueError(f"Missing genes in adata_combo gene rows: {missing}")

za = Z_genes[name_to_idx[gene_a]]
zb = Z_genes[name_to_idx[gene_b]]

# Euclidean distances from each cell to each selected gene in latent space
da = np.linalg.norm(Z_cells - za, axis=1)
db = np.linalg.norm(Z_cells - zb, axis=1)

# Raw Euclidean contrast:
score = db - da

# store score in adata_combo.obs
score_col = f"scLDM_double_score_rawdiff_{gene_a}_vs_{gene_b}"
adata_combo.obs[score_col] = np.nan
adata_combo.obs.loc[cell_mask, score_col] = score

views = [
    ("angle1", 20, 135),
    ("angle2", 20, 40),
    ("angle3", 30, 40),
    ("angle4", 10, 315),
]

outdir = Path("output/fig_5")
outdir.mkdir(parents=True, exist_ok=True)

# shared cmap/norm around 0 using raw score magnitude
max_abs = float(np.max(np.abs(score)))
norm = SignedPowerNorm(gamma=1.5, vmin=-max_abs, vmax=max_abs)

cmap = plt.get_cmap("RdBu_r")


sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

fig_cb, ax_cb = plt.subplots(figsize=(0.15, 1.5))
cbar = fig_cb.colorbar(sm, cax=ax_cb, orientation="vertical")
cbar.set_ticks([-max_abs, 0, max_abs])
cbar.set_ticklabels([f"{gene_b}-like", "neutral", f"{gene_a}-like"])
cbar.ax.tick_params(labelsize=5)

# fix vector-rendering seams
#cbar.solids.set_edgecolor("face")
#cbar.solids.set_linewidth(0)
#cbar.solids.set_rasterized(True)

fig_cb.savefig(
    outdir / f"cbar_double_score_rawdiff_{gene_a}_vs_{gene_b}.svg",
    format="svg",
    bbox_inches="tight",
    pad_inches=0.01,
    transparent=True,
    dpi=450
)
plt.close(fig_cb)

for view_name, elev, azim in views:
    fig = plt.figure(figsize=(4.5, 3.5))
    ax = fig.add_subplot(111, projection="3d")

    ax.scatter(
        Z_cells[:, 0],
        Z_cells[:, 1],
        Z_cells[:, 2],
        c=score,
        cmap=cmap,
        norm=norm,
        s=0.35,
        alpha=1,
        linewidths=0,
        depthshade=False,
        rasterized=True
    )

    # faint background gene cloud
    ax.scatter(
        Z_genes[:, 0],
        Z_genes[:, 1],
        Z_genes[:, 2],
        s=0.08,
        c="grey",
        alpha=0.08,
        linewidths=0,
        depthshade=False,
        rasterized=True
    )

    # highlight the two selected genes
    #ax.scatter(za[0], za[1], za[2], s=24, c="#d73027", marker="^", linewidths=0, depthshade=False)
    #ax.scatter(zb[0], zb[1], zb[2], s=24, c="#4575b4", marker="v", linewidths=0, depthshade=False)

    ax.view_init(elev=elev, azim=azim)

    # keep plot clean
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_zticklabels([])
    ax.tick_params(axis="x", which="major", length=2, pad=0)
    ax.tick_params(axis="y", which="major", length=2, pad=0)
    ax.tick_params(axis="z", which="major", length=2, pad=0)

    # fixed limits for cross-angle consistency
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    ax.set_zlim(-3, 3)

    fig.savefig(
        outdir / f"kang_scLDM_3d_double_score_rawdiff_{gene_a}_vs_{gene_b}_{view_name}.svg",
        dpi=600,
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.show()


## Performance across dim metrics

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FormatStrFormatter

# ------------------------------------------------------------
# Plot settings
# ------------------------------------------------------------
mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
})

# Small-panel publication settings
mpl.rcParams.update({
    "axes.linewidth": 0.6,
    "axes.labelsize": 5.5,
    "axes.titlesize": 6.0,
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "legend.fontsize": 5,
    "lines.linewidth": 1.0,
    "lines.markersize": 2.6,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "figure.dpi": 300,
    "savefig.dpi": 300,
})

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------
outdir = "output/fig_5"
os.makedirs(outdir, exist_ok=True)

# ------------------------------------------------------------
# Data loading
# ------------------------------------------------------------
results_root = (PROJECT_ROOT / "results")

dims = [2, 3, 8, 16, 32, 64]
seeds = [0, 1, 2, 3, 4]
x = list(range(len(dims)))  # categorical spacing

dataset_dirs = {
    "NeurIPS Stem Cells": "neurips_cite_stem_cells",
    "Cortex": "cortex",
    "HCA Nuclei": "hca_nuclei",
    "PBMC CITE-seq": "pbmc_cite_seq",
}

metric_keys = {
    "AUC-ROC": "auc",
    "PR-AUC": "pr_auc",
    "Max F1": "f1",
    "Poisson Deviance": "poisson_deviance",
}

def load_last_metrics(dataset_dir, dim, seed):
    path = results_root / dataset_dir / f"SCENE_{dim}D_seed{seed}" / "val_results.json"

    if not path.exists():
        raise FileNotFoundError(f"Missing results file: {path}")

    with path.open() as f:
        payload = json.load(f)

    last_metrics = {}
    for metric_name, json_key in metric_keys.items():
        if json_key not in payload:
            raise KeyError(f"Missing key '{json_key}' in {path}")
        if len(payload[json_key]) == 0:
            raise ValueError(f"Metric '{json_key}' is empty in {path}")

        last_metrics[metric_name] = payload[json_key][-1]

    return last_metrics

def build_data_from_results():
    data = {}

    for dataset_label, dataset_dir in dataset_dirs.items():
        data[dataset_label] = {metric_name: [] for metric_name in metric_keys}

        for dim in dims:
            values_by_metric = {metric_name: [] for metric_name in metric_keys}

            for seed in seeds:
                last_metrics = load_last_metrics(dataset_dir, dim, seed)

                for metric_name, value in last_metrics.items():
                    values_by_metric[metric_name].append(value)

            for metric_name in metric_keys:
                data[dataset_label][metric_name].append(values_by_metric[metric_name])

    return data

data = build_data_from_results()

# Colorblind-safe palette
colors = {
    "NeurIPS Stem Cells": "#0072B2",
    "Cortex": "#D55E00",
    "HCA Nuclei": "#009E73",
    "PBMC CITE-seq": "#CC79A7",
}

markers = {
    "NeurIPS Stem Cells": "o",
    "Cortex": "s",
    "HCA Nuclei": "D",
    "PBMC CITE-seq": "^",
}

metric_specs = {
    "AUC-ROC": {
        "ylabel": "AUC-ROC",
        "filename": "auc_roc_vs_dimension.svg",
    },
    "PR-AUC": {
        "ylabel": "PR-AUC",
        "filename": "ap_vs_dimension.svg",
    },
    "Max F1": {
        "ylabel": "Max F1",
        "filename": "micro_f1_vs_dimension.svg",
    },
    "Poisson Deviance": {
        "ylabel": "Poisson dev.",
        "filename": "poisson_deviance_vs_dimension.svg",
    },
}

zero_one_metrics = {"AUC-ROC", "PR-AUC", "Max F1"}

def metric_mean_and_sd(dataset, metric_name):
    runs = np.array(data[dataset][metric_name], dtype=float)  # dims x seeds
    mean = runs.mean(axis=1)
    sd = runs.std(axis=1, ddof=1)
    return mean, sd

def get_metric_values(metric_name):
    values = []
    for dataset in data:
        mean, sd = metric_mean_and_sd(dataset, metric_name)
        values.extend(mean - sd)
        values.extend(mean + sd)
    return values

def get_ylim_and_ticks(metric_name):
    values = get_metric_values(metric_name)

    if metric_name in zero_one_metrics:
        raw_min = min(values)
        raw_max = max(values)

        spread = raw_max - raw_min
        pad = max(0.008, 0.50 * spread)

        ymin = max(0.0, raw_min - pad)
        ymax = 1.0

        tick_start = np.ceil(ymin / 0.05) * 0.05
        yticks = np.arange(tick_start, ymax + 0.001, 0.05)

        return ymin, ymax, yticks

    ymin = min(values)
    ymax = max(values)
    yrange = ymax - ymin
    pad = yrange * 0.08 if yrange > 0 else 0.01
    return ymin - pad, ymax + pad, None

def style_axis(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="#D9D9D9", linewidth=0.4)
    ax.grid(axis="x", visible=False)
    ax.tick_params(axis="both", which="both", pad=1.5)
    ax.set_axisbelow(True)

def plot_metric(metric_name, spec, add_legend=False):
    fig, ax = plt.subplots(figsize=(1.50, 1.15), constrained_layout=True)

    for dataset in data:
        y, yerr = metric_mean_and_sd(dataset, metric_name)

        ax.errorbar(
            x,
            y,
            yerr=yerr,
            color=colors[dataset],
            marker=markers[dataset],
            label=dataset,
            markeredgewidth=0.45,
            markeredgecolor=colors[dataset],
            linewidth=1.0,
            elinewidth=0.45,
            capsize=2.0,
            capthick=0.45,
        )

    ax.set_xticks(x)
    ax.set_xticklabels([str(d) for d in dims])
    ax.set_xlabel("D", labelpad=1.5)
    ax.set_ylabel(spec["ylabel"], labelpad=1.5)
    ax.set_xlim(min(x) - 0.08, max(x) + 0.08)

    ymin, ymax, yticks = get_ylim_and_ticks(metric_name)
    ax.set_ylim(ymin, ymax)

    if yticks is not None:
        ax.set_yticks(yticks)
        ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

    style_axis(ax)

    if add_legend:
        ax.legend(
            loc="best",
            frameon=False,
            handlelength=1.4,
            borderaxespad=0.2,
            labelspacing=0.25,
        )

    outpath = os.path.join(outdir, spec["filename"])
    fig.savefig(outpath, bbox_inches="tight", transparent=True)
    plt.close(fig)
    print(f"Saved: {outpath}")

def save_standalone_legend():
    handles = [
        Line2D(
            [0], [0],
            color=colors[dataset],
            marker=markers[dataset],
            lw=1.0,
            markersize=2.8,
            label=dataset,
        )
        for dataset in data
    ]

    fig = plt.figure(figsize=(2.2, 0.42))
    fig.legend(
        handles=handles,
        labels=list(data.keys()),
        loc="center",
        ncol=5,
        frameon=False,
        handlelength=1.4,
        columnspacing=0.9,
        handletextpad=0.35,
        borderpad=0.0,
    )
    legend_path = os.path.join(outdir, "legend_datasets.svg")
    fig.savefig(legend_path, bbox_inches="tight", transparent=True)
    plt.close(fig)
    print(f"Saved: {legend_path}")

for metric_name, spec in metric_specs.items():
    plot_metric(metric_name, spec, add_legend=False)

save_standalone_legend()


In [ ]:
import pandas as pd


summary_rows = []

for dataset in data:
    for dim_idx, dim in enumerate(dims):
        row = {
            "Dataset": dataset,
            "dim": dim,
        }

        for metric_name in metric_keys:
            values = np.array(data[dataset][metric_name][dim_idx], dtype=float)
            row[f"{metric_name}_mean"] = values.mean()
            row[f"{metric_name}_std"] = values.std(ddof=1)

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(outdir, "metric_summary_by_dataset_dim.csv"), index=False)
